In [4]:
%pip install pandas openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   --------------------- ------------------ 5.8/10.8 MB 35.3 MB/s eta 0:00:01
   ---------------------------------------- 10.8/10.8 MB 33.5 MB/s  0:00:00
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ------------- -------------------------- 1/3 [pandas]
   ----------

In [1]:
import pandas as pd
import json
import os
from pathlib import Path

import torch

print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nEnvironment ready!")

Pandas: 3.1.1
PyTorch: 2.9.1+rocm7.2.1
GPU available: True
GPU: AMD Radeon(TM) 8060S Graphics

Environment ready!


In [10]:
import pandas as pd
import json
from pathlib import Path
from collections import Counter

# Lokasi fail Excel
INPUT_FILE = Path("openanswer_munsyidataset.xlsx")

# Folder output
OUTPUT_DIR = Path("malay_cleaned_dataset")
OUTPUT_DIR.mkdir(exist_ok=True)

# Nama fail output
CLEAN_DATASET = OUTPUT_DIR / "malay_evaluation_dataset.jsonl"
EXCLUDED_DATASET = OUTPUT_DIR / "malay_excluded_records.jsonl"
VALIDATION_REPORT = OUTPUT_DIR / "malay_dataset_validation_report.txt"

print("Input file exists:", INPUT_FILE.exists())
print("Output folder:", OUTPUT_DIR.resolve())

Input file exists: True
Output folder: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\malay_cleaned_dataset


In [11]:
DOMAIN_CONFIG = {
    "Campuran Kod": {
        "input": ["KORPUS UJARAN ONLINE"],
        "reference": [
            "UJARAN TIDAK FORMAL",
            "UJARAN FORMAL"
        ]
    },

    "Penukaran Kod": {
        "input": ["SOALAN"],
        "reference": [
            "JAWAPAN FORMAL",
            "JAWAPAN TIDAK FORMAL"
        ]
    },

    "Terjemahan": {
        "input": ["SOALAN (BAHASA SUMBER)"],
        "reference": [
            "JAWAPAN (BAHASA SASARAN)"
        ]
    },

    "Sentimen News": {
        "input": ["Teks"],
        "reference": ["Sifat Sentimen"],
        "max_rows": 1500
    },

    "Sentimen": {
        "input": ["SOALAN"],
        "reference": ["JAWAPAN"]
    }
}

for domain, config in DOMAIN_CONFIG.items():
    print(domain, "->", config)

Campuran Kod -> {'input': ['KORPUS UJARAN ONLINE'], 'reference': ['UJARAN TIDAK FORMAL', 'UJARAN FORMAL']}
Penukaran Kod -> {'input': ['SOALAN'], 'reference': ['JAWAPAN FORMAL', 'JAWAPAN TIDAK FORMAL']}
Terjemahan -> {'input': ['SOALAN (BAHASA SUMBER)'], 'reference': ['JAWAPAN (BAHASA SASARAN)']}
Sentimen News -> {'input': ['Teks'], 'reference': ['Sifat Sentimen'], 'max_rows': 1500}
Sentimen -> {'input': ['SOALAN'], 'reference': ['JAWAPAN']}


In [12]:
excel = pd.ExcelFile(INPUT_FILE)

print("Available sheets:")
for sheet in excel.sheet_names:
    print("-", sheet)

Available sheets:
- Campuran Kod
- Penukaran Kod
- Terjemahan
- Sentimen News
- Sentimen


In [13]:
raw_data = {}

for domain in DOMAIN_CONFIG:
    df = pd.read_excel(
        INPUT_FILE,
        sheet_name=domain,
        header=None,
        dtype=object
    )

    # Row 1 ialah header, row 2 ialah maklumat/kosong,
    # data bermula pada Excel row 3.
    headers = df.iloc[0].tolist()

    data = df.iloc[2:].copy()
    data.columns = headers
    data = data.reset_index(drop=True)

    # Had 1,500 rekod untuk Sentimen News sahaja
    if domain == "Sentimen News":
        data = data.head(1500)

    raw_data[domain] = data

    print(
        f"{domain}: {len(data)} rows "
        f"(data bermula dari Excel row 3)"
    )

Campuran Kod: 276 rows (data bermula dari Excel row 3)
Penukaran Kod: 804 rows (data bermula dari Excel row 3)
Terjemahan: 714 rows (data bermula dari Excel row 3)
Sentimen News: 1000 rows (data bermula dari Excel row 3)
Sentimen: 684 rows (data bermula dari Excel row 3)


In [16]:
for domain, df in raw_data.items():
    print(f"\n{'='*50}")
    print(domain)

    for col in df.columns:
        print(repr(col))


Campuran Kod
'KORPUS UJARAN ONLINE'
'UJARAN TIDAK FORMAL'
'UJARAN FORMAL'

Penukaran Kod
'SOALAN'
'JAWAPAN FORMAL'
'JAWAPAN TIDAK FORMAL'

Terjemahan
'SOALAN\n\n(BAHASA SUMBER)'
'JAWAPAN\n\n(BAHASA SASARAN)'

Sentimen News
'Teks'
'Sifat Sentimen'

Sentimen
'SOALAN'
'JAWAPAN'


In [17]:
def normalize_header(value):
    return " ".join(
        str(value).replace("\xa0", " ").split()
    ).strip().casefold()


for domain, df in raw_data.items():
    df.columns = [normalize_header(col) for col in df.columns]

    DOMAIN_CONFIG[domain]["input"] = [
        normalize_header(col)
        for col in DOMAIN_CONFIG[domain]["input"]
    ]

    DOMAIN_CONFIG[domain]["reference"] = [
        normalize_header(col)
        for col in DOMAIN_CONFIG[domain]["reference"]
    ]

    print(f"\n{domain}")
    print("Input:", DOMAIN_CONFIG[domain]["input"])
    print("Reference:", DOMAIN_CONFIG[domain]["reference"])


Campuran Kod
Input: ['korpus ujaran online']
Reference: ['ujaran tidak formal', 'ujaran formal']

Penukaran Kod
Input: ['soalan']
Reference: ['jawapan formal', 'jawapan tidak formal']

Terjemahan
Input: ['soalan (bahasa sumber)']
Reference: ['jawapan (bahasa sasaran)']

Sentimen News
Input: ['teks']
Reference: ['sifat sentimen']

Sentimen
Input: ['soalan']
Reference: ['jawapan']


In [21]:
cleaned_data = []
excluded_data = []
validation_summary = []

for domain, df in raw_data.items():

    config = DOMAIN_CONFIG[domain]
    required_input = config["input"]
    reference_columns = config["reference"]

    original_count = len(df)

    domain_records = []
    empty_count = 0
    incomplete_count = 0

    for excel_index, row in df.iterrows():

        excel_row = excel_index + 3

        input_values = {
            col: None if pd.isna(row[col]) or not str(row[col]).strip()
            else str(row[col]).strip()
            for col in required_input
        }

        reference_values = {
            col: None if pd.isna(row[col]) or not str(row[col]).strip()
            else str(row[col]).strip()
            for col in reference_columns
        }

        # Input wajib lengkap
        missing_input = [
            col for col, value in input_values.items()
            if value is None
        ]

        if missing_input:
            incomplete_count += 1

            excluded_data.append({
                "domain": domain,
                "excel_row": excel_row,
                "reason": "missing_input",
                "record": {
                    "input": input_values,
                    "reference": reference_values
                }
            })
            continue

        # Campuran Kod: sekurang-kurangnya satu reference wajib ada
        if domain == "Campuran Kod":

            if all(v is None for v in reference_values.values()):
                empty_count += 1

                excluded_data.append({
                    "domain": domain,
                    "excel_row": excel_row,
                    "reason": "all_references_missing",
                    "record": {
                        "input": input_values,
                        "reference": reference_values
                    }
                })
                continue

        else:
            # Domain lain: semua reference wajib lengkap
            missing_reference = [
                col for col, value in reference_values.items()
                if value is None
            ]

            if missing_reference:
                incomplete_count += 1

                excluded_data.append({
                    "domain": domain,
                    "excel_row": excel_row,
                    "reason": "incomplete_reference",
                    "missing_fields": missing_reference,
                    "record": {
                        "input": input_values,
                        "reference": reference_values
                    }
                })
                continue

        domain_records.append({
            "domain": domain,
            "input": input_values,
            "reference": reference_values,
            "source_excel_row": excel_row
        })

    # Exact duplicate removal
    seen = set()
    unique_records = []
    duplicate_count = 0

    for record in domain_records:

        signature = json.dumps(
            {
                "input": record["input"],
                "reference": record["reference"]
            },
            ensure_ascii=False,
            sort_keys=True
        )

        if signature in seen:
            duplicate_count += 1

            excluded_data.append({
                "domain": domain,
                "excel_row": record["source_excel_row"],
                "reason": "exact_duplicate",
                "record": record
            })
            continue

        seen.add(signature)
        unique_records.append(record)

    cleaned_data.extend(unique_records)

    validation_summary.append({
        "domain": domain,
        "original_rows": original_count,
        "fully_empty": empty_count,
        "incomplete": incomplete_count,
        "duplicates": duplicate_count,
        "final_cleaned": len(unique_records)
    })

display(pd.DataFrame(validation_summary))

,domain,original_rows,fully_empty,incomplete,duplicates,final_cleaned
0,Campuran Kod,276,0,1,0,275
1,Penukaran Kod,804,0,5,0,799
2,Terjemahan,714,0,0,2,712
3,Sentimen News,1000,0,0,20,980
4,Sentimen,684,0,0,2,682


In [22]:
# Export cleaned dataset
with open(CLEAN_DATASET, "w", encoding="utf-8") as f:
    for record in cleaned_data:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# Export excluded records
with open(EXCLUDED_DATASET, "w", encoding="utf-8") as f:
    for record in excluded_data:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# Generate validation report
report_lines = [
    "MALAY DATASET VALIDATION REPORT",
    "=" * 60,
    f"Total original records: {sum(x['original_rows'] for x in validation_summary)}",
    f"Total cleaned records: {len(cleaned_data)}",
    f"Total excluded records: {len(excluded_data)}",
    ""
]

for item in validation_summary:
    report_lines.extend([
        f"Domain: {item['domain']}",
        f"Original: {item['original_rows']}",
        f"Fully empty: {item['fully_empty']}",
        f"Incomplete: {item['incomplete']}",
        f"Duplicates: {item['duplicates']}",
        f"Final cleaned: {item['final_cleaned']}",
        "-" * 40
    ])

VALIDATION_REPORT.write_text(
    "\n".join(report_lines),
    encoding="utf-8"
)

print("EXPORT COMPLETED!")
print("Clean dataset:", CLEAN_DATASET.resolve())
print("Excluded records:", EXCLUDED_DATASET.resolve())
print("Validation report:", VALIDATION_REPORT.resolve())

EXPORT COMPLETED!
Clean dataset: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\malay_cleaned_dataset\malay_evaluation_dataset.jsonl
Excluded records: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\malay_cleaned_dataset\malay_excluded_records.jsonl
Validation report: C:\Users\WaRiSAN-01\TESTING_MODEL_HANIS\malay_cleaned_dataset\malay_dataset_validation_report.txt


In [23]:
print("CLEAN DATASET:", CLEAN_DATASET.exists())
print("EXCLUDED DATASET:", EXCLUDED_DATASET.exists())
print("VALIDATION REPORT:", VALIDATION_REPORT.exists())

print("\nCleaned records:", len(cleaned_data))
print("Excluded records:", len(excluded_data))

print("\nDomain distribution:")
display(
    pd.Series(
        [r["domain"] for r in cleaned_data]
    ).value_counts().rename_axis("Domain").to_frame("Records")
)

CLEAN DATASET: True
EXCLUDED DATASET: True
VALIDATION REPORT: True

Cleaned records: 3448
Excluded records: 30

Domain distribution:


,Records
Domain,
Sentimen News,980
Penukaran Kod,799
Terjemahan,712
Sentimen,682
Campuran Kod,275


In [24]:
campuran_records = [
    r for r in cleaned_data
    if r["domain"] == "Campuran Kod"
]

formal_count = sum(
    r["reference"]["ujaran formal"] is not None
    for r in campuran_records
)

informal_count = sum(
    r["reference"]["ujaran tidak formal"] is not None
    for r in campuran_records
)

both_count = sum(
    all(v is not None for v in r["reference"].values())
    for r in campuran_records
)

print("Total Campuran Kod:", len(campuran_records))
print("Formal available:", formal_count)
print("Informal available:", informal_count)
print("Both references available:", both_count)

Total Campuran Kod: 275
Formal available: 275
Informal available: 101
Both references available: 101
